In [1]:
import pandas as pd
import pyarrow.parquet as pq
import importlib

from tqdm import tqdm

import plotly.express as px
import prepare_data as prepare_data

In [2]:
PATH_FROM = "../data_clean/raw/"
PATH_TO = "../data_clean/"

# Prepare institutions and topics mappings

In [3]:
df_institutions_raw = pd.read_csv(PATH_FROM+"Maxime_institutions.csv", sep=";", decimal=",")

In [4]:
df_topics_raw = pd.read_csv(PATH_FROM+"Maxime_topics.csv", sep=";", decimal=",")

In [15]:
df_country = pd.read_csv(PATH_FROM+"Floriana_country_info.csv")

In [16]:
df_country

,name,alpha-2,alpha-3,country-code,iso_3166-2,region,sub-region,intermediate-region,region-code,sub-region-code,intermediate-region-code
0,Afghanistan,AF,AFG,4,ISO 3166-2:AF,Asia,Southern Asia,NaN,142.0,34.0,NaN
1,Åland Islands,AX,ALA,248,ISO 3166-2:AX,Europe,Northern Europe,NaN,150.0,154.0,NaN
2,Albania,AL,ALB,8,ISO 3166-2:AL,Europe,Southern Europe,NaN,150.0,39.0,NaN
3,Algeria,DZ,DZA,12,ISO 3166-2:DZ,Africa,Northern Africa,NaN,2.0,15.0,NaN
4,American Samoa,AS,ASM,16,ISO 3166-2:AS,Oceania,Polynesia,NaN,9.0,61.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
244,Wallis and Futuna,WF,WLF,876,ISO 3166-2:WF,Oceania,Polynesia,NaN,9.0,61.0,NaN
245,Western Sahara,EH,ESH,732,ISO 3166-2:EH,Africa,Northern Africa,NaN,2.0,15.0,NaN
246,Yemen,YE,YEM,887,ISO 3166-2:YE,Asia,Western Asia,NaN,142.0,145.0,NaN
247,Zambia,ZM,ZMB,894,ISO 3166-2:ZM,Africa,Sub-Saharan Africa,Eastern Africa,2.0,202.0,14.0


In [21]:
(
    df_country[["alpha-2", "name", "region"]]
    .merge(df_institutions_raw[["Country_cleaned"]].drop_duplicates(), left_on="alpha-2", right_on="Country_cleaned", how="outer")
    .dropna(subset=["Country_cleaned"])
)

,alpha-2,name,region,Country_cleaned
0,AD,Andorra,Europe,AD
1,AE,United Arab Emirates,Asia,AE
2,AF,Afghanistan,Asia,AF
3,AG,Antigua and Barbuda,Americas,AG
5,AL,Albania,Europe,AL
...,...,...,...,...
242,WS,Samoa,Oceania,WS
243,YE,Yemen,Asia,YE
245,ZA,South Africa,Africa,ZA
246,ZM,Zambia,Africa,ZM


In [22]:
219 - 193

26

In [12]:
df_institutions_raw.Country_cleaned.unique()

array(['GB', 'US', 'CZ', 'KR', 'ES', 'TT', 'SE', 'MX', 'NG', 'SK', 'FR',
       'DE', 'NL', 'PT', 'NP', 'AT', 'AR', 'EC', 'CU', 'AU', 'IE', 'SI',
       'JP', 'GH', 'BF', 'IN', 'SB', 'CN', 'ID', 'CH', 'IT', 'BR', 'RU',
       'PL', 'NO', 'GR', 'MZ', 'CA', 'PH', 'LU', 'BW', 'BE', 'SA', 'IQ',
       'EG', 'SO', 'ZW', 'SY', 'DO', 'UZ', 'NZ', 'DZ', 'HR', 'TN', 'HU',
       'TR', 'VI', 'ME', 'ZA', 'CO', 'VN', 'UG', 'BJ', 'DK', 'RO', 'CI',
       'MR', 'MA', 'UA', 'MY', 'BD', 'IR', 'ET', 'AE', 'CM', 'AO', 'HK',
       'IL', 'TW', 'FI', 'CL', 'SG', 'TH', 'MO', 'LB', 'QA', 'CY', 'LT',
       'KW', 'UY', 'KE', 'SS', 'PK', 'CR', 'EE', 'VE', 'LK', 'PR', 'LV',
       'GE', 'JM', 'BI', 'RS', 'TJ', 'BN', 'TZ', 'AZ', 'BG', 'MK', 'PE',
       'VG', 'JO', 'BA', 'MU', 'KH', 'NE', 'PS', 'CD', 'YE', 'BY', 'MM',
       'MC', 'BH', 'KZ', 'OM', 'ML', 'MN', 'GL', 'SN', 'ST', 'GA', 'CG',
       'GT', 'LI', 'IS', 'GF', 'MG', 'PA', 'LY', 'BO', 'SD', 'AL', 'RE',
       'HN', 'GP', 'AM', 'SZ', nan, 'MW', 'RW', 'LA

In [67]:
df_topics = (
    df_topics_raw
    .rename(columns={"ID": "topic_id", "Name": "topic_name",
                     "Sub_ID": "subfield_id", "Sub_Name": "subfield_name",
                     "Field_ID": "field_id", "Field_Name": "field_name",
                     "Domain_ID": "domain_id", "Domain_Name": "domain_name"})
    .drop_duplicates()
)
df_topics.to_csv(PATH_TO+"df_topics.csv", index=False)

In [6]:
df_institutions = (
    df_institutions_raw
    [["ID", "Country_cleaned"]]
    .rename(columns={"Country_cleaned": "country", "ID": "institution_id"})
    .drop_duplicates()
)

# Clean data one year

In [23]:
year = "2023"
parqToRead=PATH_FROM+"authorsPapersTopicsYearInstitutions.parquet"

columns = ["authors", "id", "year", "institution", "topic"]

table = pq.read_table(parqToRead, columns=columns, filters=[('year','=',year)])
df_year = table.to_pandas()

In [25]:
df_year.id.nunique()

4729415

In [31]:
list(set(df_year.authors))

['A5075779143',
 'A5043292005',
 'A5085830552',
 'A5049171573',
 'A5103158764',
 'A5028723452',
 'A5104203697',
 'A5056009167',
 'A5012819477',
 'A5078070740',
 'A5093546450',
 'A5008675640',
 'A5073694598',
 'A5101962389',
 'A5102796202',
 'A5018398974',
 'A5093261546',
 'A5037310068',
 'A5077953768',
 'A5033548012',
 'A5071829800',
 'A5055186794',
 'A5064041125',
 'A5024918504',
 'A5031156444',
 'A5100521119',
 'A5027578199',
 'A5013224319',
 'A5014348129',
 'A5013523535',
 'A5048554573',
 'A5021544759',
 'A5069543493',
 'A5023819217',
 'A5037990039',
 'A5008095251',
 'A5048998165',
 'A5039153063',
 'A5042685051',
 'A5066629403',
 'A5049360259',
 'A5101188625',
 'A5034218002',
 'A5019208005',
 'A5036547376',
 'A5054870369',
 'A5007033501',
 'A5100906643',
 'A5044413679',
 'A5050315491',
 'A5044283996',
 'A5050253741',
 'A5047250114',
 'A5082364495',
 'A5019568144',
 'A5101026931',
 'A5054468787',
 'A5086485401',
 'A5035975344',
 'A5060188217',
 'A5085768615',
 'A5011187627',
 'A51031

In [10]:
df_article_subfield_country = prepare_data.get_article_subfield_country(df_year,
                                                                        df_topics,
                                                                        df_institutions).dropna(subset=["country"])

In [ ]:
df_country_subfield_individual = (
    df_article_subfield_country
    .query("n_countries == 1")
    [["article_id", "country", "subfield_id"]]
    .groupby(["country", "subfield_id"], as_index=False)
    .count()
    .pivot(index="country", columns="subfield_id", values="article_id")
)

df_country_subfield_collaborative = (
    df_article_subfield_country
    .query("n_countries > 1")
    [["article_id", "country", "subfield_id"]]
    .groupby(["country", "subfield_id"], as_index=False)
    .count()
    .pivot(index="country", columns="subfield_id", values="article_id")
)

# Clean data pipeline

In [15]:
parqToRead=PATH_FROM+'authorsPapersTopicsYearInstitutions.parquet'
columns = ["authors", "id", "year", "institution", "topic"]

In [17]:
df_country_subfield_individual_list = []
df_country_subfield_collaborative_list = []

In [39]:
start_year = 1970
end_year = 2023

n_articles = 0
# authors_list = []
for year_loop in tqdm(range(start_year, end_year + 1)):
    table = pq.read_table(parqToRead, columns=columns, filters=[('year','=',str(year_loop))])
    df_year = table.to_pandas()

    n_articles += df_year.id.nunique()
    # authors_list += list(set(df_year.authors))
    
# df_global_comparison_yearly = pd.concat(df_global_comparison_list)
# df_global_comparison_yearly.to_csv(uf.PATH+f"df_global_comparison_yearly.csv")

100%|██████████| 54/54 [09:02<00:00, 10.04s/it]


In [ ]:
# len(authors_list)

115851691

In [40]:
n_articles

79093264

In [37]:
len(list(set(authors_list)))

32806409

In [ ]:
start_year = 1970
end_year = 2023

for year_loop in tqdm(range(start_year, end_year + 1)):
    table = pq.read_table(parqToRead, columns=columns, filters=[('year','=',str(year_loop))])
    df_year = table.to_pandas()

    df_article_subfield_country = prepare_data.get_article_subfield_country(df_year, df_topics, df_institutions)


    df_country_subfield_individual_list.append(
        df_article_subfield_country
        .query("n_countries == 1")
        [["article_id", "country", "subfield_id"]]
        .groupby(["country", "subfield_id"], as_index=False)
        .count()
        .pivot(index="country", columns="subfield_id", values="article_id")
        .assign(
            year = year_loop,
            mode = "individual"
        )
    )

    df_country_subfield_collaborative_list.append(
        df_article_subfield_country
        .query("n_countries > 1")
        [["article_id", "country", "subfield_id"]]
        .groupby(["country", "subfield_id"], as_index=False)
        .count()
        .pivot(index="country", columns="subfield_id", values="article_id")
        .assign(
            year = year_loop,
            mode = "collaborative"
        )
    )

df_country_subfield_yearly = (
    pd.concat(df_country_subfield_individual_list + df_country_subfield_collaborative_list)
    .reset_index()
    .set_index(["country", "year", "mode"])
)
# df_global_comparison_yearly = pd.concat(df_global_comparison_list)
# df_global_comparison_yearly.to_csv(uf.PATH+f"df_global_comparison_yearly.csv")

  0%|          | 0/44 [00:12<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
n_years = 5

df_country_subfield_rolling_yearly = (
    df_country_subfield_yearly
    .sort_index()
    .groupby(level=[0, 2])
    .rolling(n_years).sum()
    .reset_index(level=[0, 1], drop=True)
    .dropna(how="all")
)

In [ ]:
df_country_subfield_yearly.to_csv(PATH_TO+"df_ccountry_subfield.csv")
df_country_subfield_rolling_yearly.to_csv(PATH_TO+"df_country_subfield_roll_5y.csv")